
# Làm sạch nhãn OCR tiếng Việt bằng VietOCR — Kaggle

Notebook dùng **VietOCR `vgg_transformer`** để đọc lại ảnh dòng chữ và tạo nhãn đề xuất mới.

Mục tiêu:

1. giữ nguyên ảnh, split và đường dẫn ảnh;
2. không ghi đè `rec_train.txt`, `rec_val.txt`, `rec_test.txt` gốc;
3. chạy VietOCR trên ảnh thật trong `train/`, `val/`, `test/`;
4. tự động thay nhãn **train** khi VietOCR có score đủ cao;
5. với `val/test`, mặc định chỉ tạo candidate để review, tránh biến ground truth đánh giá thành pseudo-label;
6. lưu progress theo từng batch để resume khi Kaggle bị ngắt;
7. xuất audit CSV để biết dòng nào đã thay và dòng nào cần kiểm tra thủ công.

> VietOCR vẫn là một OCR model, không phải ground truth. Không nên tự động dùng prediction của model để thay toàn bộ nhãn `val/test` rồi báo cáo accuracy trên chính các nhãn đó.


## 1. Cài đặt

In [ ]:

!python -m pip install -q --upgrade pip
!python -m pip install -q gdown pandas rapidfuzz pillow tqdm
!python -m pip install -q "git+https://github.com/pbcquoc/vietocr.git@f0d24a3"

import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Hãy bật GPU accelerator trong Kaggle trước khi chạy notebook.")


## 2. Cấu hình

In [ ]:

from pathlib import Path
import json
import os
import random
import shutil
import time
import unicodedata
import zipfile

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps
from rapidfuzz.distance import Levenshtein
from tqdm.auto import tqdm
from IPython.display import display

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)

DRIVE_ID = "1_NKW1CL49NKtnT92ddaNZwGcaFJOkUgM"

ROOT = Path("/kaggle/working")
WORK = ROOT / "vietocr_dataset_cleanup"
DATA_EXTRACT = WORK / "data"
RESULTS_DIR = WORK / "results"
PROGRESS_DIR = WORK / "progress"
EXPORT_DIR = WORK / "cleaned_labels"

for p in [WORK, DATA_EXTRACT, RESULTS_DIR, PROGRESS_DIR, EXPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

VIETOCR_CONFIG = "vgg_transformer"
BATCH_SIZE = 32
AUTO_ACCEPT_SCORE = 0.90
AUTO_REPLACE_SPLITS = {"train"}
LIMIT_PER_SPLIT = None  # đặt 200 để dry-run nhanh

print("WORK:", WORK)
print("VietOCR:", VIETOCR_CONFIG)
print("Batch size:", BATCH_SIZE)
print("Auto replace threshold:", AUTO_ACCEPT_SCORE)
print("Auto replace splits:", AUTO_REPLACE_SPLITS)


## 3. Tìm hoặc giải nén dataset, kể cả nested zip

In [ ]:
def is_dataset_root(p: Path) -> bool:
    required_files = [
        p / "rec_train.txt",
        p / "rec_val.txt",
        p / "rec_test.txt",
        p / "vi_dict.txt",
    ]
    required_dirs = [p / "train", p / "val", p / "test"]
    return all(x.exists() for x in required_files) and all(x.is_dir() for x in required_dirs)


def find_dataset_roots(base: Path):
    roots = []
    if not base.exists():
        return roots
    for train_file in base.rglob("rec_train.txt"):
        p = train_file.parent
        if is_dataset_root(p):
            roots.append(p)
    return sorted(set(roots), key=lambda x: (len(x.parts), str(x)))


existing_roots = find_dataset_roots(ROOT)
if existing_roots:
    DATA_DIR = existing_roots[0]
    print("Tái sử dụng dataset đã có:", DATA_DIR)
else:
    DATA_DIR = None


def unzip_once(zip_path: Path, dest: Path):
    marker = zip_path.parent / f".{zip_path.name}.extracted"
    if marker.exists():
        return False
    print("Extract:", zip_path, "->", dest)
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dest)
    marker.touch()v
    return True


if DATA_DIR is None:
    zip_candidates = [p for p in ROOT.rglob("vi_rec_100k.zip") if p.is_file()]

    if zip_candidates:
        ZIP_PATH = sorted(zip_candidates, key=lambda p: p.stat().st_size, reverse=True)[0]
        print("Tái sử dụng zip:", ZIP_PATH)
    else:
        import gdown
        ZIP_PATH = WORK / "vi_rec_100k.zip"
        print("Không thấy zip cũ. Download từ Google Drive...")
        gdown.download(id=DRIVE_ID, output=str(ZIP_PATH), quiet=False)

    outer_marker = DATA_EXTRACT / ".outer_extracted"
    if not outer_marker.exists():
        print("Extract outer zip...")
        with zipfile.ZipFile(ZIP_PATH, "r") as zf:
            zf.extractall(DATA_EXTRACT)
        outer_marker.touch()

    while True:
        pending = []
        for nested_zip in DATA_EXTRACT.rglob("*.zip"):
            if "__MACOSX" in nested_zip.parts or nested_zip.name.startswith("._"):
                continue
            marker = nested_zip.parent / f".{nested_zip.name}.extracted"
            if not marker.exists():
                pending.append(nested_zip)

        if not pending:
            break

        for nested_zip in pending:
            unzip_once(nested_zip, nested_zip.parent)

    roots = find_dataset_roots(DATA_EXTRACT)
    if not roots:
        txts = list(DATA_EXTRACT.rglob("rec_*.txt"))[:20]
        zips = list(DATA_EXTRACT.rglob("*.zip"))[:20]
        raise FileNotFoundError(
            "Không tìm được dataset root.\n"
            f"rec_*.txt tìm thấy: {txts}\n"
            f"zip tìm thấy: {zips}"
        )
    DATA_DIR = roots[0]

TRAIN_FILE = DATA_DIR / "rec_train.txt"
VAL_FILE = DATA_DIR / "rec_val.txt"
TEST_FILE = DATA_DIR / "rec_test.txt"
DICT_FILE = DATA_DIR / "vi_dict.txt"

print("\nDATA_DIR:", DATA_DIR)
print("Train:", TRAIN_FILE)
print("Val:", VAL_FILE)
print("Test:", TEST_FILE)
print("Dict:", DICT_FILE)

## 4. Đọc label và kiểm tra đường dẫn ảnh

In [ ]:

def normalize_text(s: str) -> str:
    return unicodedata.normalize("NFC", str(s)).strip()


def read_label_file(path: Path):
    rows = []
    malformed = []
    for line_no, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
        if not line.strip():
            continue
        if "	" not in line:
            malformed.append((line_no, line))
            continue
        rel, text = line.split("	", 1)
        rows.append((rel.strip(), normalize_text(text)))
    return rows, malformed


def resolve_image(rel: str) -> Path:
    rel_path = Path(rel)
    return rel_path if rel_path.is_absolute() else DATA_DIR / rel_path


split_files = {"train": TRAIN_FILE, "val": VAL_FILE, "test": TEST_FILE}
split_rows = {}

for split, file_path in split_files.items():
    rows, malformed = read_label_file(file_path)
    if LIMIT_PER_SPLIT is not None:
        rows = rows[:LIMIT_PER_SPLIT]
    split_rows[split] = rows

    missing = [rel for rel, _ in rows if not resolve_image(rel).exists()]
    print(split, "rows =", len(rows), "| malformed =", len(malformed), "| missing =", len(missing))
    if malformed[:3]:
        print("  malformed sample:", malformed[:3])
    if missing[:3]:
        print("  missing sample:", missing[:3])

if any(not resolve_image(rel).exists() for rows in split_rows.values() for rel, _ in rows):
    raise FileNotFoundError("Có ảnh trong label không resolve được. Sửa path trước khi chạy VietOCR.")

dict_chars = set(DICT_FILE.read_text(encoding="utf-8").splitlines())
dict_chars.add(" ")

lengths = [len(text) for rows in split_rows.values() for _, text in rows]
print("Label max length:", max(lengths))
print("Label p95:", float(np.percentile(lengths, 95)))


## 5. Load VietOCR `vgg_transformer`

In [ ]:

from vietocr.tool.config import Cfg
from vietocr.tool.predictor import Predictor

config = Cfg.load_config_from_name(VIETOCR_CONFIG)
config["device"] = "cuda:0"
config["predictor"]["beamsearch"] = False

if "cnn" in config and isinstance(config["cnn"], dict):
    config["cnn"]["pretrained"] = False

detector = Predictor(config)

print("VietOCR model:", VIETOCR_CONFIG)
print("Device:", config["device"])
print("Image height:", config["dataset"]["image_height"])
print("Image min width:", config["dataset"]["image_min_width"])
print("Image max width:", config["dataset"]["image_max_width"])


## 6. Sanity check vài ảnh

In [ ]:

sample_rows = random.sample(split_rows["train"], k=min(8, len(split_rows["train"])))
preview = []

for rel, old_text in sample_rows:
    img = ImageOps.exif_transpose(Image.open(resolve_image(rel))).convert("RGB")
    pred, score = detector.predict(img, return_prob=True)

    try:
        score = float(score)
    except Exception:
        score = float(np.asarray(score, dtype=float).mean())

    pred = normalize_text(pred)
    preview.append({
        "image": rel,
        "old_text": old_text,
        "vietocr_text": pred,
        "vietocr_score": score,
        "edit_similarity": float(Levenshtein.normalized_similarity(old_text, pred)),
    })

display(pd.DataFrame(preview))



## 7. Inference toàn bộ dataset với resume

Mỗi split lưu prediction ngay vào JSONL. Nếu Kaggle ngắt, chạy lại cell này sẽ bỏ qua ảnh đã xong.
Notebook dùng `predict_batch` của VietOCR và fallback từng ảnh nếu một batch gặp lỗi/OOM.


In [ ]:
def score_to_float(score):
    if score is None:
        return float("nan")
    try:
        return float(score)
    except Exception:
        arr = np.asarray(score, dtype=float)
        return float(arr.mean())


def load_done(progress_file: Path):
    done = {}
    if not progress_file.exists():
        return done
    for line in progress_file.read_text(encoding="utf-8").splitlines():
        if line.strip():
            obj = json.loads(line)
            done[obj["image"]] = obj
    return done


def predict_split(split: str, rows):
    progress_file = PROGRESS_DIR / f"{split}_vietocr_predictions.jsonl"
    done = load_done(progress_file)
    pending = [(rel, text) for rel, text in rows if rel not in done]

    print(f"[{split}] total={len(rows)} done={len(done)} pending={len(pending)}")
    if not pending:
        return done

    with progress_file.open("a", encoding="utf-8") as fout:
        for start in tqdm(range(0, len(pending), BATCH_SIZE), desc=f"VietOCR {split}"):
            batch_rows = pending[start:start + BATCH_SIZE]
            images = []
            valid_rows = []

            for rel, old_text in batch_rows:
                try:
                    img = ImageOps.exif_transpose(Image.open(resolve_image(rel))).convert("RGB")
                    images.append(img)
                    valid_rows.append((rel, old_text))
                except Exception as e:
                    obj = {
                        "split": split,
                        "image": rel,
                        "old_text": old_text,
                        "vietocr_text": "",
                        "vietocr_score": None,
                        "error": f"IMAGE_LOAD_ERROR: {e}",
                    }
                    fout.write(json.dumps(obj, ensure_ascii=False) + "\n")
                    done[rel] = obj

            if not images:
                fout.flush()
                continue

            try:
                preds, scores = detector.predict_batch(images, return_prob=True)
            except Exception:
                torch.cuda.empty_cache()
                preds, scores = [], []
                for img in images:
                    try:
                        pred, score = detector.predict(img, return_prob=True)
                    except Exception:
                        pred, score = "", float("nan")
                    preds.append(pred)
                    scores.append(score)

            for (rel, old_text), pred, score in zip(valid_rows, preds, scores):
                obj = {
                    "split": split,
                    "image": rel,
                    "old_text": old_text,
                    "vietocr_text": normalize_text(pred),
                    "vietocr_score": score_to_float(score),
                    "error": None,
                }
                fout.write(json.dumps(obj, ensure_ascii=False) + "\n")
                done[rel] = obj

            fout.flush()

    return done


all_predictions = {}
start_time = time.time()

for split in ["train", "val", "test"]:
    all_predictions[split] = predict_split(split, split_rows[split])

print("Total inference minutes:", (time.time() - start_time) / 60)

## 8. Phân loại: thay tự động / giữ nguyên / cần review

In [ ]:

def chars_supported(text: str) -> bool:
    return all(ch in dict_chars for ch in text)


def classify_row(split, old_text, pred_text, score, error):
    old_n = normalize_text(old_text)
    pred_n = normalize_text(pred_text)

    if error:
        return "REVIEW_ERROR"
    if pred_n == old_n:
        return "KEEP_SAME"
    if not pred_n:
        return "REVIEW_EMPTY"
    if not chars_supported(pred_n):
        return "REVIEW_UNSUPPORTED_CHAR"
    if score is None or not np.isfinite(score):
        return "REVIEW_NO_SCORE"
    if split not in AUTO_REPLACE_SPLITS:
        return "REVIEW_EVAL"
    if score >= AUTO_ACCEPT_SCORE:
        return "REPLACE_HIGH_SCORE"
    return "REVIEW_LOW_SCORE"


audit_rows = []

for split in ["train", "val", "test"]:
    pred_map = all_predictions[split]

    for rel, old_text in split_rows[split]:
        obj = pred_map.get(rel)
        if obj is None:
            audit_rows.append({
                "split": split,
                "image": rel,
                "old_text": old_text,
                "vietocr_text": "",
                "vietocr_score": np.nan,
                "edit_similarity": np.nan,
                "action": "REVIEW_MISSING_PREDICTION",
                "new_text": old_text,
            })
            continue

        pred_text = normalize_text(obj.get("vietocr_text", ""))
        score = obj.get("vietocr_score", None)
        error = obj.get("error", None)
        sim = float(Levenshtein.normalized_similarity(old_text, pred_text)) if pred_text else 0.0
        action = classify_row(split, old_text, pred_text, score, error)
        new_text = pred_text if action == "REPLACE_HIGH_SCORE" else old_text

        audit_rows.append({
            "split": split,
            "image": rel,
            "old_text": old_text,
            "vietocr_text": pred_text,
            "vietocr_score": score,
            "edit_similarity": sim,
            "action": action,
            "new_text": new_text,
        })

audit_df = pd.DataFrame(audit_rows)
audit_path = RESULTS_DIR / "vietocr_cleanup_audit.csv"
audit_df.to_csv(audit_path, index=False, encoding="utf-8-sig")

display(audit_df.groupby(["split", "action"]).size().rename("count").reset_index())
print("Audit:", audit_path)


## 9. Xuất label cleaned + VietOCR raw candidate

In [ ]:
def write_label_file(path: Path, rows):
    with path.open("w", encoding="utf-8") as f:
        for rel, text in rows:
            f.write(f"{rel}\t{normalize_text(text)}\n")


for split in ["train", "val", "test"]:
    sub = audit_df[audit_df["split"] == split].copy()

    cleaned_path = EXPORT_DIR / f"rec_{split}_cleaned.txt"
    write_label_file(cleaned_path, list(zip(sub["image"], sub["new_text"])))

    raw_rows = []
    for _, row in sub.iterrows():
        raw_text = row["vietocr_text"] if row["vietocr_text"] else row["old_text"]
        raw_rows.append((row["image"], raw_text))

    raw_path = EXPORT_DIR / f"rec_{split}_vietocr_raw.txt"
    write_label_file(raw_path, raw_rows)

    print(split)
    print("  cleaned:", cleaned_path)
    print("  raw candidate:", raw_path)

review_df = audit_df[audit_df["action"].str.startswith("REVIEW")].copy()
review_df = review_df.sort_values(
    ["split", "vietocr_score", "edit_similarity"],
    ascending=[True, False, True],
)
review_path = RESULTS_DIR / "review_needed.csv"
review_df.to_csv(review_path, index=False, encoding="utf-8-sig")
print("\nReview file:", review_path, "| rows:", len(review_df))

## 10. Thống kê mức độ khác biệt

In [ ]:

summary = []
for split in ["train", "val", "test"]:
    sub = audit_df[audit_df["split"] == split]
    total = len(sub)
    differs = int((sub["old_text"] != sub["vietocr_text"]).sum())
    replaced = int((sub["action"] == "REPLACE_HIGH_SCORE").sum())
    review = int(sub["action"].str.startswith("REVIEW").sum())

    summary.append({
        "split": split,
        "total": total,
        "vietocr_differs_from_original": differs,
        "vietocr_differs_pct": 100 * differs / max(total, 1),
        "auto_replaced": replaced,
        "auto_replaced_pct": 100 * replaced / max(total, 1),
        "need_review": review,
        "need_review_pct": 100 * review / max(total, 1),
        "mean_edit_similarity": sub["edit_similarity"].mean(),
        "mean_vietocr_score": sub["vietocr_score"].mean(),
    })

summary_df = pd.DataFrame(summary)
display(summary_df)
summary_df.to_csv(RESULTS_DIR / "cleanup_summary.csv", index=False, encoding="utf-8-sig")


## 11. Xem các thay đổi lớn, score cao

In [ ]:

largest_changes = (
    audit_df[
        (audit_df["old_text"] != audit_df["vietocr_text"]) &
        (audit_df["vietocr_text"].str.len() > 0)
    ]
    .sort_values(["vietocr_score", "edit_similarity"], ascending=[False, True])
    .head(30)
)

display(largest_changes[[
    "split", "image", "old_text", "vietocr_text",
    "vietocr_score", "edit_similarity", "action"
]])


## 12. Hiển thị ảnh của một số dòng cần review

In [ ]:
import matplotlib.pyplot as plt

to_show = review_df.head(12)
for _, row in to_show.iterrows():
    image_path = resolve_image(row["image"])
    img = ImageOps.exif_transpose(Image.open(image_path)).convert("RGB")

    plt.figure(figsize=(14, 2.5))
    plt.imshow(img)
    plt.axis("off")
    plt.title(
        f"{row['split']} | score={row['vietocr_score']:.4f} | sim={row['edit_similarity']:.4f}\n"
        f"OLD: {row['old_text']}\n"
        f"VIETOCR: {row['vietocr_text']}",
        loc="left",
        fontsize=9,
    )
    plt.show()


## 13. Tuỳ chọn: cho phép auto-replace cả val/test

**Không khuyến nghị cho báo cáo đánh giá chính thức.**

Nếu đã kiểm tra bằng mắt và muốn VietOCR thay cả validation/test, đổi:

```python
AUTO_REPLACE_SPLITS = {"train", "val", "test"}
```

rồi chạy lại từ phần **8. Phân loại** trở xuống. Không cần inference lại.


## 14. Đóng gói kết quả

In [ ]:

package_dir = WORK / "vietocr_cleanup_package"
if package_dir.exists():
    shutil.rmtree(package_dir)

(package_dir / "labels").mkdir(parents=True)
(package_dir / "results").mkdir(parents=True)

for p in EXPORT_DIR.glob("*.txt"):
    shutil.copy2(p, package_dir / "labels" / p.name)

for name in ["vietocr_cleanup_audit.csv", "review_needed.csv", "cleanup_summary.csv"]:
    src = RESULTS_DIR / name
    if src.exists():
        shutil.copy2(src, package_dir / "results" / name)

metadata = {
    "seed": SEED,
    "vietocr_config": VIETOCR_CONFIG,
    "batch_size": BATCH_SIZE,
    "auto_accept_score": AUTO_ACCEPT_SCORE,
    "auto_replace_splits": sorted(AUTO_REPLACE_SPLITS),
    "data_dir": str(DATA_DIR),
    "note": "Val/test cleaned files preserve original labels by default; review candidates manually.",
}

(package_dir / "metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

zip_path = shutil.make_archive(str(WORK / "vietocr_dataset_cleanup"), "zip", root_dir=package_dir)
print("Package:", zip_path)
print("Audit:", RESULTS_DIR / "vietocr_cleanup_audit.csv")
print("Review:", RESULTS_DIR / "review_needed.csv")
print("Cleaned labels:", EXPORT_DIR)
